In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/amananandrai/ag-news-classification-dataset/train.csv
/kaggle/input/datasets/amananandrai/ag-news-classification-dataset/test.csv


# Imports

In [4]:
import os
import random
import numpy as np
import pandas as pd
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


# Load and Preprocess Data

In [5]:
DATA_PATH = "/kaggle/input/datasets/amananandrai/ag-news-classification-dataset/"

train_df = pd.read_csv(DATA_PATH + "train.csv")
test_df  = pd.read_csv(DATA_PATH + "test.csv")

print("Dataset shape:", train_df.shape)
train_df.head()

Dataset shape: (120000, 3)


,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


## Check for Nulls

In [6]:
print(f"\nNull values: {train_df.isnull().sum()},\n\n{test_df.isnull().sum()}")


Null values: Class Index    0
Title          0
Description    0
dtype: int64,

Class Index    0
Title          0
Description    0
dtype: int64


## Check Duplicates

In [7]:
print("Duplicate rows:", train_df.duplicated().sum(), test_df.duplicated().sum())

Duplicate rows: 0 0


## Concatenate Title + Description

In [8]:
train_df["text"] = train_df["Title"] + " [SEP] " + train_df["Description"]
test_df["text"] = test_df["Title"] + " [SEP] " + test_df["Description"]

train_df.head()

,Class Index,Title,Description,text
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...","Oil prices soar to all-time record, posing new..."


## Check and Correct Length for GPU Efficiency

In [9]:
train_df["word_len"] = train_df["Description"].str.split().apply(len)
print(train_df["word_len"].describe())

count    120000.000000
mean         31.060508
std           9.760460
min           1.000000
25%          25.000000
50%          30.000000
75%          36.000000
max         173.000000
Name: word_len, dtype: float64


In [10]:
train_df[train_df["word_len"] <= 3 ]

,Class Index,Title,Description,text,word_len
17812,2,Sabres agree to terms with 2003 first-round pi...,#NAME?,Sabres agree to terms with 2003 first-round pi...,1
21983,4,Intel looks to fend off AMD with new 2006 chipset,&lt;strong&gt;Exclusive&lt;/strong&gt; Blackfo...,Intel looks to fend off AMD with new 2006 chip...,3
22955,2,Top of 3rd,#NAME?,Top of 3rd [SEP] #NAME?,1
23969,2,"Blues re-sign D Backman, four others",#NAME?,"Blues re-sign D Backman, four others [SEP] #NAME?",1
27805,2,Wild re-sign D Schultz,#NAME?,Wild re-sign D Schultz [SEP] #NAME?,1
27806,2,"Blue Jackets re-signs G Denis, RW Vyborny",#NAME?,"Blue Jackets re-signs G Denis, RW Vyborny [SEP...",1
29104,2,Predators re-sign D Zidlicky,#NAME?,Predators re-sign D Zidlicky [SEP] #NAME?,1
38557,4,So what is it about Win2k security MS &lt;u&gt...,Expect clarification soonest,So what is it about Win2k security MS &lt;u&gt...,3
45162,4,From soup to nuts with Microsoft #8217;s colla...,&lt;strong&gt;Interview&lt;/strong&gt; Anoop G...,From soup to nuts with Microsoft #8217;s colla...,3
45922,2,"- UMPIRES: Home,Andy Fletcher; First, Tim Welk...",#NAME?,"- UMPIRES: Home,Andy Fletcher; First, Tim Welk...",1


### Remove "#Name?" string

In [11]:
len(train_df[train_df["text"].str.contains("#NAME\\?", regex=True)])

24

In [12]:
train_df = train_df[~train_df["text"].str.contains("#NAME\\?", regex=True)].reset_index(drop=True)
train_df = train_df[~train_df["Description"].str.contains("#NAME\\?", regex=True)].reset_index(drop=True)

# for test also
test_df = test_df[~test_df["text"].str.contains("#NAME\\?", regex=True)].reset_index(drop=True)
test_df = test_df[~test_df["Description"].str.contains("#NAME\\?", regex=True)].reset_index(drop=True)

### Removing HTML Text

In [13]:
import html
train_df["text"] = train_df["text"].apply(html.unescape)
train_df["Description"] = train_df["Description"].apply(html.unescape)

# for test also
test_df["text"] = test_df["text"].apply(html.unescape)
test_df["Description"] = test_df["Description"].apply(html.unescape)

In [14]:
#special characters
train_df[train_df["Description"].str.contains(r"[^a-zA-Z0-9\s,.\-!?':;\"()&]", regex=True, na=False)]

,Class Index,Title,Description,text,word_len
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...,12
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...,30
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...,29
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...,27
5,3,"Stocks End Up, But Near Year Lows (Reuters)",Reuters - Stocks ended slightly higher on Frid...,"Stocks End Up, But Near Year Lows (Reuters) [S...",30
...,...,...,...,...,...
119957,3,Murdoch will shell out \$44 mil. for Manhattan...,"NEW YORK -- Rupert Murdoch, the billionaire CE...",Murdoch will shell out \$44 mil. for Manhattan...,42
119958,1,AU Says Sudan Begins Troop Withdrawal from Dar...,Reuters - The African Union said on Saturday t...,AU Says Sudan Begins Troop Withdrawal from Dar...,26
119959,1,Insurgents Attack Iraq Election Offices (Reuters),Reuters - Insurgents have launched attacks\on ...,Insurgents Attack Iraq Election Offices (Reute...,26
119960,1,Syria Redeploys Some Security Forces in Lebano...,"Reuters - Syria, under intense pressure to qui...",Syria Redeploys Some Security Forces in Lebano...,28


In [15]:
rows = train_df[train_df["Description"].str.contains(r"[^a-zA-Z0-9\s,.\-!?':;\"()&\\]", regex=True, na=False)]
# inspecting first element
print(rows.iloc[2]["Description"])

Forbes.com - After earning a PH.D. in Sociology, Danny Bazil Riley started to work as the general manager at a commercial real estate firm at an annual base salary of  #36;70,000. Soon after, a financial planner stopped by his desk to drop off brochures about insurance benefits available through his employer. But, at 32, "buying insurance was the furthest thing from my mind," says Riley.


In [16]:
# starting with # but skips #+digits and word boundaries
train_df.loc[train_df["Description"].str.contains(r"#(?!\d+\b)", regex=True, na=False)].head()

,Class Index,Title,Description,text,word_len
26828,4,SD in the cards for Nokia,"<a href=""http://news.zdnet.co.uk/hardware/mobi...","SD in the cards for Nokia [SEP] <a href=""http:...",10
27065,4,SD in the cards for Nokia,"<a href=""http://p2pnet.net/story/2429"">Nokia t...","SD in the cards for Nokia [SEP] <a href=""http:...",9
27721,4,SD in the cards for Nokia,"<a href=""http://news.com.com/NokiajoinsSecureD...","SD in the cards for Nokia [SEP] <a href=""http:...",11
28032,4,Nokia overcomes SD Card phobia,"<a href=""http://www.technewsworld.com/story/No...","Nokia overcomes SD Card phobia [SEP] <a href=""...",12
28179,2,Ballpark scuffle says more about fans than pla...,#39;You suck! You #39;re a piece of amp;\$#*...,Ballpark scuffle says more about fans than pla...,39


In [17]:
rows = train_df.loc[train_df["Description"].str.contains(r"#(?!\d+\b)", regex=True, na=False)]

print(rows.iloc[0]["Description"])

<a href="http://news.zdnet.co.uk/hardware/mobile/0,39020360,39166545,00.htm">Nokia gives SD cards its blessing</a> <font size=-1 color=#6f6f6f><nobr>ZDNet.co.uk</nobr>


### Remove HTML Tags

In [18]:
import re
train_df["text"] = train_df["text"].apply(lambda x: re.sub(r"<[^>]+>", "", x))
train_df["Description"] = train_df["Description"].apply(lambda x: re.sub(r"<[^>]+>", "", x))

# for test also
test_df["text"] = test_df["text"].apply(lambda x: re.sub(r"<[^>]+>", "", x))
test_df["Description"] = test_df["Description"].apply(lambda x: re.sub(r"<[^>]+>", "", x))

In [19]:
rows = train_df.loc[train_df["Description"].str.contains(r"#(?!\d+\b)", regex=True, na=False)]

print(rows.iloc[7]["Description"])

\\(Second term was beginning.)\\ What happen ?\\ Somebody set up us the bomb.\\ We get Ohio exit poll.\\ What you say?!\\ Fox News turn on.\\ It's you !!\\ How are you gentlemen !!?\\ All your ballots are belong to us.\\ You are on the way to election defeat.\\ What you say !!?\\ You have no chance to win make your time.\\ Ha Ha Ha Ha ....\\ take off, every zogby\\ for great justice\\Thanks coderman and #inforanarchy!\\


In [20]:
# remove backslashes
train_df["Description"] = train_df["Description"].str.replace(r"\\+", " ", regex=True)

# remove excessive dots
train_df["Description"] = train_df["Description"].str.replace(r"\.{2,}", ".", regex=True)

# -------------------- in "text" column also
# remove backslashes
train_df["text"] = train_df["text"].str.replace(r"\\+", " ", regex=True)

# remove excessive dots
train_df["text"] = train_df["text"].str.replace(r"\.{2,}", ".", regex=True)


# ------------------------for test also-------------------------------#

# remove backslashes
test_df["Description"] = test_df["Description"].str.replace(r"\\+", " ", regex=True)

# remove excessive dots
test_df["Description"] = test_df["Description"].str.replace(r"\.{2,}", ".", regex=True)

# -------------------- in "text" column also
# remove backslashes
test_df["text"] = test_df["text"].str.replace(r"\\+", " ", regex=True)

# remove excessive dots
test_df["text"] = test_df["text"].str.replace(r"\.{2,}", ".", regex=True)

In [21]:
rows = train_df.loc[train_df["Description"].str.contains(r"#(?!\d+\b)", regex=True, na=False)]

print(rows.iloc[7]["Description"])

 (Second term was beginning.)  What happen ?  Somebody set up us the bomb.  We get Ohio exit poll.  What you say?!  Fox News turn on.  It's you !!  How are you gentlemen !!?  All your ballots are belong to us.  You are on the way to election defeat.  What you say !!?  You have no chance to win make your time.  Ha Ha Ha Ha .  take off, every zogby  for great justice Thanks coderman and #inforanarchy! 


### now count words in "text" column

In [22]:
train_df["word_len"] = train_df["text"].str.split().apply(len)
print(train_df["word_len"].describe())

count    119976.000000
mean         38.907898
std          10.137720
min           7.000000
25%          33.000000
50%          38.000000
75%          44.000000
max         178.000000
Name: word_len, dtype: float64


### Set Max Length

In [23]:
print("95th percentile length:",
      train_df["word_len"].quantile(0.95))

95th percentile length: 54.0


In [24]:
MAX_LEN = 64 # because most sequences are less than 54 words
train_df = train_df[train_df["word_len"] <= MAX_LEN].reset_index(drop=True)
len(train_df)

117594

## Check Class Balance

In [25]:
class_counts = train_df["Class Index"].value_counts().sort_index()
print(class_counts)

Class Index
1    29217
2    29372
3    29780
4    29225
Name: count, dtype: int64


In [26]:
min_count = train_df["Class Index"].value_counts().min()

train_df = train_df.groupby("Class Index", group_keys=False).sample(n=min_count, random_state=SEED).reset_index(drop=True)

In [27]:
class_counts = train_df["Class Index"].value_counts().sort_index()
print(class_counts)

Class Index
1    29217
2    29217
3    29217
4    29217
Name: count, dtype: int64


## Tokenization

In [28]:
# Pytorch expects lables to start from 0 and onwards
train_df["Class Index"] = train_df["Class Index"].astype("category").cat.codes
test_df["Class Index"] = test_df["Class Index"].astype("category").cat.codes

In [29]:
train_labels = train_df["Class Index"]
train_texts = train_df["text"]

# for test also
test_labels = test_df["Class Index"]
test_texts = test_df["text"]

In [30]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    max_length=MAX_LEN
)

test_encodings = tokenizer(
    test_texts.tolist(),
    truncation=True,
    max_length=MAX_LEN
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## Pytorch Dataset

In [31]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
test_dataset = Dataset(test_encodings, test_labels)

### DataLoader (Generator)

In [32]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer)

In [33]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=data_collator, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, collate_fn=data_collator, num_workers=4, pin_memory=True)

# Training

In [32]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4 # total 4 classes
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

num_training_steps = len(train_loader) * 3  # epochs=3
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0.1 * num_training_steps,
    num_training_steps=num_training_steps
)

In [34]:
# send model to GPU
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [34]:
# Tracking using tensorboard
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter("runs/bert_agnews")

2026-03-04 01:11:40.262711: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772586700.476039      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772586700.535131      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772586701.063294      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772586701.063330      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772586701.063333      55 computation_placer.cc:177] computation placer alr

In [44]:
%reload_ext tensorboard
%tensorboard --logdir runs

Reusing TensorBoard on port 6006 (pid 123), started 0:19:42 ago. (Use '!kill 123' to kill it.)

<IPython.core.display.Javascript object>

In [37]:
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm

scaler = GradScaler()
global_step = 0
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}

        with autocast():
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

        # TensorBoard
        writer.add_scalar("Train/Loss", loss.item(), global_step)
        writer.add_scalar("Train/LR", scheduler.get_last_lr()[0], global_step)

        global_step += 1
        
        loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}]")
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    writer.add_scalar("Epoch/Avg_Train_Loss", avg_loss, epoch)

/tmp/ipykernel_576/2220973503.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


  0%|          | 0/7305 [00:00<?, ?it/s]

/tmp/ipykernel_576/2220973503.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  0%|          | 0/7305 [00:00<?, ?it/s]

  0%|          | 0/7305 [00:00<?, ?it/s]

## Saving and downloading model for future use

In [39]:
from transformers import AutoModelForSequenceClassification

save_path = "/kaggle/working/bert_agnews_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /kaggle/working/bert_agnews_model


In [40]:
import shutil
shutil.make_archive("/kaggle/working/bert_agnews_model", 'zip', save_path)

'/kaggle/working/bert_agnews_model.zip'

# Evaluation

## Loading model again from downloads

In [40]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import os

model_path = os.path.abspath("/kaggle/input/models/muhammadsafeer12348/multi-class-bert-news-classifier/pytorch/default/1")
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

model.to(device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [47]:
model.eval()

correct = 0
total = 0
test_loss = 0
counter = 0

with torch.no_grad():
    for batch in test_loader:
        counter += 1
        if counter % 100 == 0:
            print(f"Batch{counter} of {len(test_loader)}")
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits

        test_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

avg_test_loss = test_loss / len(test_loader)
test_acc = correct / total

writer.add_scalar("test/Loss", avg_test_loss)
writer.add_scalar("test/Accuracy", test_acc)

print(f"test Loss: {avg_test_loss:.4f} | test Accuracy: {test_acc:.4f}")

Batch100 of 475
Batch200 of 475
Batch300 of 475
Batch400 of 475
test Loss: 0.5106 | test Accuracy: 0.9426


In [ ]:
writer.close()

# Tensorboard